### Python Insights - Analisando Dados com Python
#### Case - Previsão de Cancelamento de Clientes (Banco Nordic Crédito)

O Banco Nordic Crédito, instituição financeira de médio porte com operação em três países (França, Espanha e Alemanha), vem enfrentando uma taxa de cancelamento de clientes acima do esperado nos últimos trimestres. A diretoria de retenção precisa entender quais características dos clientes mais se relacionam com o cancelamento, para direcionar campanhas de retenção de forma mais assertiva e reduzir o custo de aquisição de novos clientes.

Base de dados: Churn Modelling Dataset - Kaggle 

In [1]:
# Passo a passo do projeto
# Bibliotecas utilizadas: pandas, openpyxl, nbformat, ipykernel, plotly

# Passo 1: Abrir a base de dados (Importar a base de dados)
# Passo 2: Vizualizar a base de dados
    # entender as informações disponiveis
    # possiveis problemas/erros na base de dados
# Passo 3: Corrigir problemas da base de dados (tratamento de dados)
    # valores em formatos errados
    # informações vazias
# Passo 4: Analise inicial (entender quantos clientes cancelaram)
# Passo 5: Analise detalhada (causa de cancelamentos dos clientes, como toda coluna impacta no cancelamento)
    # entender se existe alguma faixa etária mais propensa a cancelar
    # entender se os clientes com score de crédito mais baixo cancelam com mais frequência
    # entender se a localização geográfica (França, Espanha, Alemanha) influencia a taxa de cancelamento
    # entender se os clientes inativos têm uma probabilidade maior de encerrar o relacionamento com o banco
    # entender se o número de produtos contratados (conta, cartão, empréstimo, etc.) impacta a fidelização
    # entender se ter cartão de crédito reduz as chances de cancelamento
# Passo 6: sugerir ações práticas de retenção, direcionadas aos perfis de clientes com maior risco de saída.

In [2]:
# Passo 1: Abrir a base de dados (Importar a base de dados)
import pandas as pd 

tabela = pd.read_csv('Churn_Modelling.csv')

In [3]:
# Passo 2: Vizualizar a base de dados

tabela = tabela.drop(columns=['RowNumber','CustomerId','Surname'])

display(tabela)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
# Passo 3: Corrigir problemas da base de dados (tratamento de dados)
tabela.info()
tabela.duplicated().sum()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  str    
 2   Gender           10000 non-null  str    
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), str(2)
memory usage: 859.5 KB


np.int64(0)

In [5]:
# Passo 4: Analise inicial (entender quantos clientes cancelaram)

# Quantidade - contar quantos clientes são 0 e quantos são 1 na coluna Exited

display(tabela['Exited'].value_counts())

# Porcentagem

display(tabela['Exited'].value_counts(normalize=True))

Exited
0    7963
1    2037
Name: count, dtype: int64

Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64

#### Faixa etária mais propensa a cancelar

In [6]:
# Passo 5: Analise detalhada (causa de cancelamentos dos clientes, como toda coluna impacta no cancelamento)
import plotly.express as px

tabela['faixa_etaria'] = pd.cut(tabela['Age'], bins=[18,30,40,50,60,100], 
                                  labels=['18-30','31-40','41-50','51-60','60+'])

grafico = px.histogram(tabela, x='faixa_etaria', color='Exited',
                        barnorm ='percent', text_auto='.1f')
grafico.show()

grafico.write_image('imagens/faixa_etaria_x_cancelamento.png')

**Hipótese:** existe uma faixa etária mais propensa a cancelar.

**Conclusão:** confirma-se que a faixa de 51-60 anos apresenta a maior taxa de cancelamento 
(56,2%), bem acima das demais faixas. Chama atenção que o padrão não é simplesmente 
crescente com a idade: a taxa sobe de 18-30 até 51-60, mas cai novamente na faixa 60+ 
(24,8%), sugerindo que o fator por trás do cancelamento é específico dessa faixa 
intermediária, não da idade avançada em geral. A causa ainda não foi identificada.

#### Score de crédito

In [7]:
tabela['faixa_score'] = pd.cut(tabela['CreditScore'], bins=[300,500,650,750,850],                   
                                labels=['300-500','501-650','651-750','751-850'])

grafico = px.histogram(tabela, x='faixa_score', color='Exited',
                            barnorm='percent', text_auto='.1f')

grafico.show()

**Hipótese:** clientes com score de crédito mais baixo cancelam com mais frequência.

**Conclusão:** o score de crédito parece ter pouca influência isolada sobre a decisão de cancelamento, a faixa de menor de score (300-500) tem a 
maior taxa de cancelamento (23,6%), mas a diferença entre as faixas é pequena (variando 
de 19,2% a 23,6%).

#### Geografia

In [8]:
grafico = px.histogram(tabela, x='Geography', color='Exited',
                        barnorm='percent', text_auto='.1f')

grafico.show()
grafico.write_image('imagens/geografia_x_cancelamento.png')

**Hipótese:** a localização geográfica do cliente influencia a taxa de cancelamento.

**Conclusão:** confirma-se — a Alemanha apresenta uma taxa de cancelamento de 32,4%, quase 
o dobro de França (16,2%) e Espanha (16,7%), que têm taxas bem próximas entre si.


#### Clientes ativos

In [9]:
grafico = px.histogram(tabela, x='IsActiveMember', color='Exited',
                        barnorm='percent', text_auto='.1f')

grafico.show()

grafico.write_image('imagens/atividade_x_cancelamento.png')

**Hipótese:** clientes inativos (`IsActiveMember = 0`) cancelam com mais frequência que os ativos.

**Conclusão:** confirma-se com força, clientes inativos cancelam quase o dobro (26,9%) 
comparado aos ativos (14,3%).

#### Produtos contratos

In [10]:
grafico = px.histogram(tabela, x='NumOfProducts', color='Exited',
                        barnorm='percent', text_auto='.1f')

grafico.show()

grafico.write_image('imagens/produtos_x_cancelamento.png')

**Hipótese:** quanto mais produtos um cliente contrata, maior a chance de cancelamento.

**Conclusão:** a hipótese se confirma parcialmente e de forma acentuada, clientes com 
3 ou mais produtos apresentam taxas de cancelamento maiores (82,7% e 100%, 
respectivamente) comparados a quem tem 1-2 produtos. Embora o grupo de 4 produtos seja 
pequeno (60 clientes), o padrão já é visível e consistente a partir de 3 produtos 
(266 clientes), sugerindo que ter muitos produtos pode estar associado a insatisfação, 
sobrecarga de tarifas, ou complexidade de gestão da conta.

#### Cartão de credito

In [11]:
grafico = px.histogram(tabela, x='HasCrCard', color='Exited',
                        barnorm='percent',text_auto='.1f')

grafico.show()

**Hipótese:** ter cartão de crédito reduz as chances de cancelamento.

**Conclusão:** a hipótese não se confirma, a diferença entre quem tem cartão (20,2% 
de cancelamento) e quem não tem (20,8%) é praticamente nula. Essa variável não parece 
ter nenhuma influência relevante na decisão de cancelamento.

#### Tempo como cliente

In [12]:
display(tabela['Tenure'].describe())

grafico = px.histogram(tabela, x='Tenure', color='Exited',
                       barnorm='percent', text_auto='.1f')

grafico.show()

count    10000.000000
mean         5.012800
std          2.892174
min          0.000000
25%          3.000000
50%          5.000000
75%          7.000000
max         10.000000
Name: Tenure, dtype: float64

**Hipótese:** clientes com menos tempo de conta (`Tenure` baixo) cancelam com mais frequência 
que clientes mais antigos.

**Conclusão:** a taxa de cancelamento se mantém relativamente estável (entre 17% e 23%) em todos os anos de relacionamento com o banco, sem uma tendência clara de queda ou aumento conforme o tempo de conta avança. 
O tempo como cliente não parece ser um fator relevante isoladamente.

#### Balance (saldo em conta)

In [13]:
tabela['faixa_balance'] = pd.cut(tabela['Balance'], bins=[-1, 0, 50000, 100000, 150000, 300000],
                                    labels=['Zerado','0-50','50-100k','100-150k','150k+'])

grafico = px.histogram(tabela, x='faixa_balance', color='Exited',
                        barnorm='percent', text_auto='.1f')

grafico.show()

**Hipótese:** clientes com saldo zerado em conta têm maior probabilidade de cancelamento 
(possível indício de conta inativa na prática).

**Conclusão:** a hipótese não confirma, na verdade, o oposto acontece: clientes com 
saldo zerado apresentam a menor taxa de cancelamento (13,8%), enquanto clientes com saldo 
baixo mas não-zero (0-50k) têm a maior taxa (34,7%). Isso sugere que saldo zerado pode 
representar um tipo específico de produto ou perfil de cliente (talvez conta sem 
movimentação de crédito, mas ainda satisfeito com o banco), enquanto valores baixos e 
positivos podem indicar clientes insatisfeitos ou com dificuldades financeiras, mais 
propensos a encerrar o relacionamento.

#### Estimativa de salario

In [14]:
tabela['faixa_salario'] = pd.cut(tabela['EstimatedSalary'], bins=[0, 50000, 100000, 150000, 200000],
                                    labels=['0-50k','50-100k','100-150k','150-200k'])

grafico = px.histogram(tabela, x='faixa_salario', color='Exited',
                        barnorm='percent', text_auto='.1f')
grafico.show()

**Hipótese:** o salário estimado do cliente influencia a taxa de cancelamento.

**Conclusão:** a taxa de cancelamento é praticamente idêntica em todas as faixas salariais (variando entre 19,9% e 21,5%). O salário estimado não parece 
ter nenhuma relação com a decisão de cancelamento.

#### Faixa etaria x numero de produtos

In [15]:
display(tabela['NumOfProducts'].value_counts())


display(tabela.groupby('faixa_etaria', observed=True)['NumOfProducts'].mean())



NumOfProducts
1    5084
2    4590
3     266
4      60
Name: count, dtype: int64

faixa_etaria
18-30    1.556012
31-40    1.534262
41-50    1.528017
51-60    1.464241
60+      1.508621
Name: NumOfProducts, dtype: float64

#### Faixa etária x número de produtos

**Hipótese:** clientes mais velhos cancelam por terem mais produtos contratados.

**Conclusão:** os dados não confirmam essa relação, a média de produtos é, na verdade, 
similar (e até levemente menor) na faixa de maior cancelamento. A causa do pico de 
cancelamento entre 51-60 anos parece estar ligada a outro fator, ainda não identificado.

#### Faixa etaria x Cliente ativo

In [16]:
tabela.groupby('faixa_etaria', observed=True)['IsActiveMember'].mean()

faixa_etaria
18-30    0.510277
31-40    0.497641
41-50    0.471552
51-60    0.578419
60+      0.808190
Name: IsActiveMember, dtype: float64

**Hipótese:** a faixa etária de 51-60 anos cancela mais porque tem uma proporção menor 
de clientes ativos.

**Conclusão:** os dados não confirmam essa relação, a faixa 51-60 tem, na verdade, uma 
proporção de clientes ativos (57,8%) maior que 41-50, 31-40 e 18-30, e menor apenas que 
60+ (80,8%). Chama atenção que a faixa 60+ tem a maior proporção de ativos de todas, mas 
cancela bem menos que 51-60, o que reforça que inatividade não é o fator explicativo 
por trás do pico de cancelamento nessa faixa etária específica. Ainda a causa não foi identificada.

#### Geografia x Saldo em conta

In [17]:
tabela.groupby('Geography')['Balance'].apply(lambda x: (x==0).mean())

Geography
France     0.482250
Germany    0.000000
Spain      0.484053
Name: Balance, dtype: float64

**Hipótese:** a diferença de cancelamento entre os países está relacionada à distribuição 
de saldo em conta.

**Conclusão:** confirma-se de forma marcante , na Alemanha, 0% dos clientes têm saldo 
zerado, contra ~48% em França e Espanha. Como contas com saldo zerado apresentam a menor 
taxa de cancelamento (13,8%), a ausência desse perfil de conta na Alemanha pode explicar 
parte significativa da sua taxa de cancelamento mais alta (32,4%). Isso sugere que a 
diferença de cancelamento entre os países pode estar mais ligada à composição de produtos 
bancários oferecidos por região do que a um comportamento cultural do cliente alemão.

#### Faixa etária x Geografia

In [18]:
tabela.groupby(['faixa_etaria', 'Geography'], observed=True)['Exited'].mean()

faixa_etaria  Geography
18-30         France       0.048685
              Germany      0.125280
              Spain        0.084746
31-40         France       0.095197
              Germany      0.209393
              Spain        0.093023
41-50         France       0.281162
              Germany      0.488201
              Spain        0.273043
51-60         France       0.525886
              Germany      0.695473
              Spain        0.459893
60+           France       0.199134
              Germany      0.389381
              Spain        0.208333
Name: Exited, dtype: float64

**Hipótese:** o pico de cancelamento na faixa 51-60 anos está concentrado especificamente 
na Alemanha.

**Conclusão:** o pico de cancelamento na faixa 51-60 ocorre de forma consistente nos três países (França: 52,6%, Alemanha: 69,5%, 
Espanha: 45,9%), e a Alemanha mantém uma taxa mais alta que os demais países em todas as faixas etárias, não apenas na 51-60. Isso indica que idade e geografia são dois fatores de risco independentes, que se somam (o cliente alemão de 51-60 anos concentra o maior risco combinado, com quase 70% de cancelamento), mas nenhum explica o outro. 
A causa raiz específica do pico etário ainda não foi identificada.

#### Geografia x Membro Ativo

In [19]:
tabela.groupby('Geography', observed=True)['IsActiveMember'].mean()

Geography
France     0.516753
Germany    0.497409
Spain      0.529673
Name: IsActiveMember, dtype: float64

**Conclusão:** a proporção de clientes ativos é semelhante entre os três países, descartando 
a atividade do cliente como explicação para a maior taxa de cancelamento na Alemanha, 
reforçando que a diferença de composição de produtos (saldo zerado) é o fator mais provável.

#### Número de produtos x Membro ativo

In [20]:
tabela.groupby('NumOfProducts')['IsActiveMember'].mean()

NumOfProducts
1    0.504131
2    0.532898
3    0.424812
4    0.483333
Name: IsActiveMember, dtype: float64

**Conclusão:** a diferença de atividade entre os grupos de produtos é pequena, descartando 
a inatividade como principal explicação para o alto cancelamento em clientes com 3+ produtos.

In [21]:
# Passo 6: sugerir ações práticas de retenção...

### Passo 6: Conclusões e Recomendações

#### Principais fatores de risco identificados

A análise mostrou quatro fatores com relações claras e consistentes aos cancelamentos:

1. **Número de produtos contratados** o fator mais forte de todos. Clientes com 3 ou 
mais produtos cancelam entre 82,7% e 100% das vezes, contra apenas 7,6% entre quem tem 2 
produtos.
2. **Geografia** clientes na Alemanha cancelam quase o dobro (32,4%) comparado a França 
e Espanha (~16%). A causa provável não é comportamental, e sim estrutural: 0% das contas 
alemãs têm saldo zerado, contra ~48% em França e Espanha, e contas com saldo zerado são 
justamente as que menos cancelam.
3. **Atividade do cliente** clientes inativos cancelam quase o dobro (26,9%) comparado 
aos ativos (14,3%).
4. **Faixa etária** clientes entre 51-60 anos apresentam a maior taxa de cancelamento 
(56,2%), um padrão presente nos três países, mas a causa exata não foi identificada 
mesmo após testar produtos e atividade como possíveis explicações.

#### Segmento de maior risco

O cruzamento entre faixa etária e geografia revelou o grupo mais crítico: **clientes 
alemães entre 51-60 anos, com quase 70% de taxa de cancelamento** o maior risco 
combinado encontrado em toda a análise.

#### Recomendações de ação

- **Revisar a estrutura de produtos para clientes com 3+ contratos:** investigar se há 
sobrecarga de tarifas, atendimento fragmentado ou complexidade de gestão, e testar uma 
oferta consolidada (pacote único com desconto) para esse segmento.
- **Avaliar a introdução, na Alemanha, de um produto equivalente à conta com saldo zerado** 
disponível em França e Espanha, monitorando se isso reduz o cancelamento local.
- **Criar campanhas de reengajamento para clientes inativos** já que a inatividade quase 
dobra a chance de cancelamento, um alerta automático após um período sem uso poderia 
ajudar no problema.
- **Priorizar ações de retenção para o segmento alemão de 51-60 anos**, mesmo sem uma causa 
raiz totalmente identificada, dado o alto risco combinado desse grupo.

#### Limitações da análise

A causa específica do pico de cancelamento na faixa 51-60 anos não foi identificada com 
as informações disponíveis, produtos contratados e nível de atividade foram testados e 
descartados como explicação. Investigações futuras poderiam explorar dados adicionais, 
como motivo declarado de cancelamento ou histórico de atendimento ao cliente, não 
disponíveis neste dataset.